# Solutions to notebook 1: Getting started: symbols instead of numbers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/sympy-for-engineers/blob/main/solutions/01_getting_started.ipynb)

Worked solutions to the exercises at the end of [notebook 1](../notebooks/01_getting_started.ipynb).
Try each exercise yourself before reading its solution: the struggle is where the learning happens.

In [1]:
import sympy as sp

sp.init_printing(use_latex="mathjax")

## Exercise 1

Expand $(v + \Delta v)^2 - v^2$ and factor the result.

In [2]:
v, dv = sp.symbols("v Delta_v", positive=True)

expanded = sp.expand((v + dv) ** 2 - v**2)
factored = sp.factor(expanded)
expanded, factored

⎛  2                        ⎞
⎝Δᵥ  + 2⋅Δᵥ⋅v, Δᵥ⋅(Δᵥ + 2⋅v)⎠

In [3]:
# verify: the hand result, and one numerical case (v = 3, dv = 0.5 gives 3.5**2 - 3**2 = 3.25)
assert sp.simplify(factored - dv * (2 * v + dv)) == 0
assert factored.subs({v: 3, dv: sp.Rational(1, 2)}) == sp.Rational(13, 4)

**Answer.** Expanded: $2v\,\Delta v + \Delta v^2$. Factored: $\Delta v\,(2v + \Delta v)$.
The $v^2$ terms cancel, so for a small change the difference is close to $2v\,\Delta v$.

## Exercise 2

Solve Bernoulli's equation $P_1 + \tfrac12\rho v_1^2 = P_2 + \tfrac12\rho v_2^2$ for $v_2$.
Why does SymPy return one solution when all symbols are positive and two when they are not?

In [4]:
# all symbols positive
P1, P2, rho, v1, v2 = sp.symbols("P_1 P_2 rho v_1 v_2", positive=True)
bernoulli = sp.Eq(P1 + rho * v1**2 / 2, P2 + rho * v2**2 / 2)

positive_solutions = sp.solve(bernoulli, v2)
positive_solutions

⎡   _____________________⎤
⎢  ╱                   2 ⎥
⎢╲╱  2⋅P₁ - 2⋅P₂ + ρ⋅v₁  ⎥
⎢────────────────────────⎥
⎣           √ρ           ⎦

In [5]:
# the same equation with symbols that carry no assumptions
P1a, P2a, rhoa, v1a, v2a = sp.symbols("P_1 P_2 rho v_1 v_2")
bernoulli_any = sp.Eq(P1a + rhoa * v1a**2 / 2, P2a + rhoa * v2a**2 / 2)

general_solutions = sp.solve(bernoulli_any, v2a)
general_solutions

⎡      _____________________       _____________________⎤
⎢     ╱                   2       ╱                   2 ⎥
⎢    ╱  2⋅P₁ - 2⋅P₂ + ρ⋅v₁       ╱  2⋅P₁ - 2⋅P₂ + ρ⋅v₁  ⎥
⎢-  ╱   ─────────────────── ,   ╱   ─────────────────── ⎥
⎣ ╲╱             ρ            ╲╱             ρ          ⎦

In [6]:
v2_expr = positive_solutions[0]

# verify: substitute back into the equation, and check the limit of equal pressures (v2 = v1)
assert len(positive_solutions) == 1 and len(general_solutions) == 2
assert sp.simplify(bernoulli.lhs - bernoulli.rhs.subs(v2, v2_expr)) == 0
assert sp.simplify(v2_expr.subs(P2, P1) - v1) == 0

# numbers: water, v1 = 2 m/s, pressure falls by 10 kPa
float(v2_expr.subs({P1: 150e3, P2: 140e3, rho: 998, v1: 2}))        # m/s

4.9030684433648934

**Answer.** $v_2 = \sqrt{v_1^2 + \dfrac{2\,(P_1 - P_2)}{\rho}}$.

The equation is quadratic in $v_2$, so it has the two roots $\pm\sqrt{\dots}$. When $v_2$ is
declared positive, SymPy discards the negative root because it contradicts the assumption.
Without assumptions it cannot tell which root you want and returns both. With water at 2 m/s
and a 10 kPa pressure drop the speed rises to about 4.9 m/s.

## Exercise 3

For the sand grain above, find the radius at which $Re_p = 0.1$.
Hint: substitute `vt_expr` into `Re`, then use `sp.solve`.

In [7]:
# the formulas of the lesson
rho, v, D, mu = sp.symbols("rho v D mu", positive=True)
Rp, rho_p, rho_f, g, vt = sp.symbols("R_p rho_p rho_f g v_t", positive=True)

Re = rho * v * D / mu
weight = sp.Rational(4, 3) * sp.pi * Rp**3 * (rho_p - rho_f) * g
drag = 6 * sp.pi * mu * Rp * vt
vt_expr = sp.solve(sp.Eq(weight, drag), vt)[0]

# particle Reynolds number: fluid density, settling velocity, particle diameter
Re_p = Re.subs({rho: rho_f, v: vt_expr, D: 2 * Rp})
Re_p

    3                  
4⋅Rₚ ⋅g⋅ρ_f⋅(-ρ_f + ρₚ)
───────────────────────
            2          
         9⋅μ           

In [8]:
# put in the numbers for everything except the radius, then solve
sand = {rho_p: 2650, rho_f: 998, mu: 1.0e-3, g: 9.81}

Rp_num = sp.solve(sp.Eq(Re_p.subs(sand), 0.1), Rp)[0]
Rp_num        # m

2.40505110320616e-5

In [9]:
# verify 1: the hand formula R_p**3 = 9 mu**2 Re_p / (4 rho_f (rho_p - rho_f) g)
Rp_hand = (9 * 1.0e-3**2 * 0.1 / (4 * 998 * (2650 - 998) * 9.81)) ** (1 / 3)
assert abs(Rp_num / Rp_hand - 1) < 1e-9

# verify 2: recompute the settling velocity and the Reynolds number for this radius
vt_num = vt_expr.subs(sand).subs(Rp, Rp_num)
Re_check = 998 * vt_num * 2 * Rp_num / 1.0e-3
assert abs(Re_check - 0.1) < 1e-9
print(f"R_p = {float(Rp_num) * 1e6:.1f} µm, v_t = {float(vt_num) * 1e3:.2f} mm/s, Re_p = {float(Re_check):.3f}")

R_p = 24.1 µm, v_t = 2.08 mm/s, Re_p = 0.100


**Answer.** $Re_p = \dfrac{4\,\rho_f(\rho_p - \rho_f)\,g\,R_p^3}{9\mu^2}$, so
$R_p = \left(\dfrac{9\mu^2 Re_p}{4\,\rho_f(\rho_p-\rho_f)\,g}\right)^{1/3} \approx 2.4\times10^{-5}$ m.
That is a radius of about 24 µm (a 48 µm grain), which settles at about 2.1 mm/s. Because
$Re_p \propto R_p^3$, halving the radius of the lesson's grain cuts $Re_p$ from 0.9 to about 0.1.

## Exercise 4

Print the ideal gas law solved for $V$ as LaTeX.

In [10]:
P, V, n, R, T = sp.symbols("P V n R T", positive=True)
ideal_gas = sp.Eq(P * V, n * R * T)

V_expr = sp.solve(ideal_gas, V)[0]
print(sp.latex(V_expr))                  # the right-hand side only
print(sp.latex(sp.Eq(V, V_expr)))        # the whole equation

\frac{R T n}{P}
V = \frac{R T n}{P}


In [11]:
# verify: put V back into the gas law, and check one mole at 273.15 K and 1 atm (22.4 L)
assert sp.simplify(ideal_gas.lhs.subs(V, V_expr) - ideal_gas.rhs) == 0
V_molar = V_expr.subs({n: 1, R: 8.314, T: 273.15, P: 101325})
assert abs(V_molar - 0.0224) < 1e-4
V_molar        # m^3

0.0224127224278312

**Answer.** `sp.latex` prints `\frac{R T n}{P}`, and for the whole equation `V = \frac{R T n}{P}`,
which renders as $V = \dfrac{R T n}{P}$. SymPy orders the symbols alphabetically, so edit the
string by hand if you prefer $nRT/P$.